In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score


def resolve_data_dir() -> Path:
    """Locate project `data/` folder (checks cwd then parent for data/dane1.txt)."""
    cwd = Path.cwd().resolve()
    for root in (cwd, cwd.parent):
        if (root / "data" / "dane1.txt").is_file():
            return root / "data"
    return cwd / "data"

In [ ]:
# load a daneXX.txt file - each line is "input output"
def load_dane(filepath):
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                data.append([float(parts[0]), float(parts[1])])
    data = np.array(data)
    return data[:, 0].reshape(-1, 1), data[:, 1]

In [ ]:
# Dane files live under data/ (see resolve_data_dir)
DATA_DIR = resolve_data_dir()
datasets = {
    "dane1": DATA_DIR / "dane1.txt",
    "dane2": DATA_DIR / "dane2.txt",
}

results = {}

for name, filepath in datasets.items():
    print(f"\n=== Dataset: {name} ===")
    X, y = load_dane(filepath)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # linear model
    lin_model = LinearRegression()
    lin_model.fit(X_train, y_train)
    y_pred_lin = lin_model.predict(X_test)
    r2_lin = r2_score(y_test, y_pred_lin)

    # polynomial model degree 3
    poly_model = make_pipeline(PolynomialFeatures(degree=3), LinearRegression())
    poly_model.fit(X_train, y_train)
    y_pred_poly = poly_model.predict(X_test)
    r2_poly = r2_score(y_test, y_pred_poly)

    print(f"  LinearRegression    R2 (test): {r2_lin:.6f}")
    print(f"  PolynomialFeatures  R2 (test): {r2_poly:.6f}")

    if r2_poly > r2_lin:
        print(f"  -> Polynomial model fits better for {name}")
    else:
        print(f"  -> Linear model fits better (or equal) for {name}")

    results[name] = (r2_lin, r2_poly, X, y, lin_model, poly_model)

In [ ]:
# plot comparison for both datasets
fig, axes = plt.subplots(1, len(results), figsize=(12, 5))

for ax, (name, (r2_lin, r2_poly, X, y, lin_m, poly_m)) in zip(axes, results.items()):
    x_range = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)

    ax.scatter(X, y, s=20, color='gray', label='data', alpha=0.6)
    ax.plot(x_range, lin_m.predict(x_range), color='blue', linewidth=2, label=f'Linear R²={r2_lin:.4f}')
    ax.plot(x_range, poly_m.predict(x_range), color='red', linestyle='--', linewidth=2, label=f'Poly(3) R²={r2_poly:.4f}')
    ax.set_title(name)
    ax.legend(fontsize=9)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.suptitle("Linear vs Polynomial Regression Comparison")
plt.tight_layout()
plt.show()